# Day 2 Lab - Incident Simulator & Log Rate Analysis
### AIOps Fundamentals Training | Microland

---

## What This Notebook Does

This notebook generates a **4-hour log stream** simulating a normal, busy IT operations
environment — and then injects a **burst of 500 failed authentication events** at exactly
the 2-hour mark, simulating a brute-force attack against a domain controller.

The challenge: the 500 anomalous events are hidden inside a stream of **3,500 normal events**
covering authentication, service changes, process activity, and network connections.
Without AIOps, finding this needle in the haystack means manually reading 4,000 log lines.
With Elastic's **Log Rate Analysis**, it takes seconds.

---

## Learning Objectives
- Understand how a realistic mixed log stream looks — normal noise plus a hidden incident
- Use Log Rate Analysis to identify the statistically overrepresented signal at the incident time
- Create an automated alert rule that opens a Case (simulated ITSM ticket) when the pattern is detected
- Reflect on the difference between manual investigation and ML-assisted root cause analysis

---

## Scenario Background

**Time:** A normal Tuesday afternoon in Microland's IT operations centre.
**Environment:** 10 Windows servers, 3 domain controllers, standard background event traffic.
**At Hour 2:** An external IP begins a brute-force attack against the 'administrator' account
on domain controller DC-PROD-01, firing 500 failed logon attempts in 8 minutes.
**Your job:** Use Log Rate Analysis to find it, then automate the alert.

---

## Before You Start
1. Elastic Cloud deployment running (from Day 1 lab)
2. Cloud ID and API Key ready
3. Running in Google Colab

> **No Python knowledge required.** Run each cell in order using the Play button or Shift+Enter.


## Step 1 - Install Required Libraries


In [ ]:
!pip install elasticsearch --quiet
print('Libraries installed.')


## Step 2 - Configuration

Paste your Elastic Cloud credentials below.
The simulation parameters are pre-set to match the lab scenario — you can adjust them
after completing the main lab to experiment with different injection sizes and timings.


In [ ]:
# -----------------------------------------------------------
# ELASTIC CLOUD CREDENTIALS
# -----------------------------------------------------------
CLOUD_ID   = 'Your Cloud ID'
API_KEY    = 'Your API Key'
INDEX_NAME = 'aiops-lab-lograte-simulator'

# -----------------------------------------------------------
# SIMULATION PARAMETERS
# -----------------------------------------------------------
TOTAL_HOURS          = 4      # Total log stream duration in hours
NORMAL_EVENT_COUNT   = 3500   # Background noise events across the 4 hours
INJECTION_COUNT      = 500    # Brute-force failed logins to inject
INJECTION_HOUR       = 2.0    # Hour at which the attack begins
INJECTION_DURATION_M = 8      # Attack duration in minutes
ATTACK_IP            = '91.108.4.201'    # Simulated attacker IP (Telegram infra range)
ATTACK_TARGET_USER   = 'administrator'   # Target account
ATTACK_TARGET_HOST   = 'DC-PROD-01'      # Target domain controller

print('Configuration set.')
print(f'   Index           : {INDEX_NAME}')
print(f'   Log stream      : {TOTAL_HOURS} hours')
print(f'   Normal events   : {NORMAL_EVENT_COUNT:,}')
print(f'   Injected events : {INJECTION_COUNT:,} brute-force attempts')
print(f'   Attack window   : hour {INJECTION_HOUR} for {INJECTION_DURATION_M} minutes')
print(f'   Attacker IP     : {ATTACK_IP}')
print(f'   Target account  : {ATTACK_TARGET_USER} on {ATTACK_TARGET_HOST}')


## Step 3 - Connect to Elastic Cloud


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)
info = es.info()
print(f'Connected to cluster: {info["cluster_name"]}')
print(f'   Version: {info["version"]["number"]}')


## Step 4 - Create the Index

The schema below extends the Day 1 log schema with additional ECS fields
relevant to security and network events — giving Log Rate Analysis more signal to work with.


In [ ]:
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')

mapping = {
    'mappings': {
        'properties': {
            '@timestamp':          {'type': 'date'},
            'event.action':        {'type': 'keyword'},
            'event.category':      {'type': 'keyword'},
            'event.outcome':       {'type': 'keyword'},
            'event.code':          {'type': 'keyword'},
            'event.dataset':       {'type': 'keyword'},
            'host.name':           {'type': 'keyword'},
            'host.os.name':        {'type': 'keyword'},
            'user.name':           {'type': 'keyword'},
            'source.ip':           {'type': 'ip'},
            'source.port':         {'type': 'integer'},
            'source.geo.country':  {'type': 'keyword'},
            'network.protocol':    {'type': 'keyword'},
            'process.name':        {'type': 'keyword'},
            'log.level':           {'type': 'keyword'},
            'message':             {'type': 'text'},
            'is_injected':         {'type': 'boolean'},
            'injection_label':     {'type': 'keyword'}
        }
    }
}

es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index "{INDEX_NAME}" created with extended ECS-aligned mapping.')


## Step 5 - Generate the 4-Hour Log Stream

### Normal Event Mix (3,500 events across 4 hours)

| Event Type | Count | Description |
|------------|-------|-------------|
| Successful logons (4624) | 900 | Normal user and service account logins |
| Failed logons - normal (4625) | 300 | Occasional fat-finger or expired password |
| Service start/stop (7036) | 700 | Services cycling - routine operational activity |
| Process creation (4688) | 600 | New processes - admin and application activity |
| Network connections | 600 | Internal DNS, HTTP, SMB traffic |
| Config change events | 400 | Group policy, registry, scheduled task changes |

### Injected Attack Events (500 events in 8 minutes)

| Event Type | Count | Description |
|------------|-------|-------------|
| **Brute-force failed logons (4625)** | **500** | Rapid-fire failed logins from external IP 91.108.4.201 against administrator account |

> **The signal-to-noise ratio:** 500 attack events hidden in 3,500 normal events.
> Ratio of 1:7. Without ML, this requires reading 4,000 log lines to find.


In [ ]:
import random
from datetime import datetime, timedelta, timezone

random.seed(99)

now        = datetime.now(timezone.utc)
start_time = now - timedelta(hours=TOTAL_HOURS)
inj_start  = start_time + timedelta(hours=INJECTION_HOUR)
inj_end    = inj_start  + timedelta(minutes=INJECTION_DURATION_M)

# Reference data
SERVERS  = ['WINTEL-SRV-01', 'WINTEL-SRV-02', 'WINTEL-SRV-03', 'WINTEL-SRV-04',
            'DC-PROD-01', 'DC-PROD-02', 'DC-PROD-03',
            'APP-SRV-01', 'APP-SRV-02', 'FILE-SRV-01']
INT_IPS  = ['10.10.1.10', '10.10.1.11', '10.10.1.25', '10.10.2.5',
            '192.168.1.50', '192.168.1.51', '192.168.10.20', '10.10.3.100']
USERS    = ['jsmith', 'aparikh', 'nkumar', 'rdesai', 'srao',
            'admin_it', 'svc_backup', 'svc_monitor', 'svc_patching', 'mgupta']
PROCS    = ['svchost.exe', 'lsass.exe', 'explorer.exe', 'powershell.exe',
            'cmd.exe', 'mmc.exe', 'taskmgr.exe', 'wuauclt.exe', 'msiexec.exe']
SVCS     = ['Windows Update', 'Print Spooler', 'DHCP Client', 'DNS Client',
            'Windows Defender', 'Remote Desktop Services', 'Task Scheduler',
            'Windows Management Instrumentation', 'Server']
PROTOCOLS = ['dns', 'smb', 'http', 'ldap', 'kerberos', 'rdp']
COUNTRIES = ['India', 'India', 'India', 'India', 'Singapore', 'US']

def rand_ts(t0, t1):
    return t0 + timedelta(seconds=random.uniform(0, (t1 - t0).total_seconds()))

def base_doc(ts, action, category, outcome, code, dataset, host, user, ip, level, msg):
    return {
        '@timestamp':         ts.isoformat(),
        'event.action':       action,
        'event.category':     category,
        'event.outcome':      outcome,
        'event.code':         code,
        'event.dataset':      dataset,
        'host.name':          host,
        'host.os.name':       'Windows Server 2019',
        'user.name':          user,
        'source.ip':          ip,
        'source.port':        random.randint(49152, 65535),
        'source.geo.country': random.choice(COUNTRIES),
        'network.protocol':   random.choice(PROTOCOLS),
        'process.name':       '',
        'log.level':          level,
        'message':            msg,
        'is_injected':        False,
        'injection_label':    'normal'
    }

events = []

# --- Normal logon success (900) ---
for _ in range(900):
    u, ip, h = random.choice(USERS), random.choice(INT_IPS), random.choice(SERVERS)
    ts = rand_ts(start_time, now)
    e = base_doc(ts, 'logged-in', 'authentication', 'success', '4624',
                 'system.security', h, u, ip, 'information',
                 f'Account successfully logged on. Account: {u}, Source: {ip}')
    events.append(e)

# --- Normal logon failures (300) - spread evenly ---
for _ in range(300):
    u, ip, h = random.choice(USERS), random.choice(INT_IPS), random.choice(SERVERS)
    ts = rand_ts(start_time, now)
    e = base_doc(ts, 'authentication_failure', 'authentication', 'failure', '4625',
                 'system.security', h, u, ip, 'warning',
                 f'Account failed to log on. Account: {u}, Source: {ip}, Reason: Wrong password')
    events.append(e)

# --- Service start/stop (700) ---
for _ in range(700):
    svc, h = random.choice(SVCS), random.choice(SERVERS)
    state  = random.choice(['started', 'stopped'])
    ts     = rand_ts(start_time, now)
    e = base_doc(ts, f'service-{state}', 'process', 'success', '7036',
                 'system.system', h, 'SYSTEM', '127.0.0.1', 'information',
                 f'The {svc} service entered the {state} state.')
    events.append(e)

# --- Process creation (600) ---
for _ in range(600):
    proc, u, h = random.choice(PROCS), random.choice(USERS), random.choice(SERVERS)
    ts = rand_ts(start_time, now)
    e = base_doc(ts, 'process_started', 'process', 'success', '4688',
                 'system.security', h, u, random.choice(INT_IPS), 'information',
                 f'New process created. Name: {proc}, Creator: {u}')
    e['process.name'] = proc
    events.append(e)

# --- Network connections (600) ---
for _ in range(600):
    proto, h = random.choice(PROTOCOLS), random.choice(SERVERS)
    ip       = random.choice(INT_IPS)
    ts       = rand_ts(start_time, now)
    e = base_doc(ts, f'{proto}-connection', 'network', 'success', '5156',
                 'system.security', h, random.choice(USERS), ip, 'information',
                 f'Network connection allowed. Protocol: {proto}, Source: {ip}')
    e['network.protocol'] = proto
    events.append(e)

# --- Config changes (400) ---
change_actions = ['group-policy-update', 'registry-change',
                  'scheduled-task-created', 'audit-policy-change']
for _ in range(400):
    action, h = random.choice(change_actions), random.choice(SERVERS)
    u  = random.choice(['admin_it', 'svc_patching', 'SYSTEM'])
    ts = rand_ts(start_time, now)
    e = base_doc(ts, action, 'configuration', 'success', '4719',
                 'system.security', h, u, random.choice(INT_IPS), 'information',
                 f'System configuration change: {action} by {u}')
    events.append(e)

# --- INJECTED: Brute-force attack (500 in 8 minutes) ---
for i in range(INJECTION_COUNT):
    ts = rand_ts(inj_start, inj_end)
    events.append({
        '@timestamp':         ts.isoformat(),
        'event.action':       'authentication_failure',
        'event.category':     'authentication',
        'event.outcome':      'failure',
        'event.code':         '4625',
        'event.dataset':      'system.security',
        'host.name':          ATTACK_TARGET_HOST,
        'host.os.name':       'Windows Server 2019',
        'user.name':          ATTACK_TARGET_USER,
        'source.ip':          ATTACK_IP,
        'source.port':        random.randint(49152, 65535),
        'source.geo.country': 'Netherlands',
        'network.protocol':   'smb',
        'process.name':       '',
        'log.level':          'warning',
        'message':            (
            f'Account failed to log on. '
            f'Account: {ATTACK_TARGET_USER}, Source: {ATTACK_IP}, '
            f'Reason: Unknown user name or bad password'
        ),
        'is_injected':        True,
        'injection_label':    'brute_force_attack'
    })

events.sort(key=lambda x: x['@timestamp'])

normal_ct   = sum(1 for e in events if not e['is_injected'])
injected_ct = sum(1 for e in events if     e['is_injected'])

print(f'Generated {len(events):,} total log events')
print(f'   Normal events   : {normal_ct:,}')
print(f'   Injected attack : {injected_ct:,} events in {INJECTION_DURATION_M}-minute window')
print(f'   Signal:Noise    : 1:{normal_ct // injected_ct}')
print(f'   Attack window   : {inj_start.strftime("%H:%M:%S")} to {inj_end.strftime("%H:%M:%S")} UTC')
print(f'   Attacker IP     : {ATTACK_IP} (geolocation: Netherlands)')
print(f'   Target          : {ATTACK_TARGET_USER} on {ATTACK_TARGET_HOST}')


## Step 6 - Visualise the Log Stream

Before uploading, plot the event volume over time.
This shows exactly what the Log Rate Analysis tool will be looking for:
a sudden, sustained spike in event count in a specific category.

> **Observation:** The authentication failure count at the 2-hour mark should spike sharply.
> However, when all event types are combined, the spike may be less obvious —
> exactly the real-world challenge that makes manual investigation so difficult.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from collections import defaultdict

# Bin events into 5-minute buckets
bucket_size_min = 5
buckets_normal  = defaultdict(int)
buckets_attack  = defaultdict(int)

for e in events:
    ts  = datetime.fromisoformat(e['@timestamp'].replace('Z', '+00:00'))
    idx = int((ts - start_time).total_seconds() / (bucket_size_min * 60))
    if e['is_injected']:
        buckets_attack[idx] += 1
    else:
        buckets_normal[idx] += 1

max_bucket = max(max(buckets_normal), max(buckets_attack))
x      = list(range(max_bucket + 1))
x_mins = [i * bucket_size_min for i in x]
y_norm = [buckets_normal.get(i, 0) for i in x]
y_atk  = [buckets_attack.get(i, 0) for i in x]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top panel: all events
ax1.bar(x_mins, [n + a for n, a in zip(y_norm, y_atk)],
        width=bucket_size_min * 0.9, color='#1B3A6B', alpha=0.7, label='All events')
ax1.bar(x_mins, y_atk,
        width=bucket_size_min * 0.9, color='#C55A11', alpha=0.9, label='Attack events')
ax1.axvline(INJECTION_HOUR * 60, color='#C55A11', linestyle='--', linewidth=2)
ax1.set_ylabel('Events per 5-min bucket', fontsize=10)
ax1.set_title('Total Event Volume (all types) — Attack partially visible in combined view',
              fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Bottom panel: authentication failures only
auth_fail_normal = defaultdict(int)
auth_fail_attack = defaultdict(int)
for e in events:
    if e['event.action'] == 'authentication_failure':
        ts  = datetime.fromisoformat(e['@timestamp'].replace('Z', '+00:00'))
        idx = int((ts - start_time).total_seconds() / (bucket_size_min * 60))
        if e['is_injected']:
            auth_fail_attack[idx] += 1
        else:
            auth_fail_normal[idx] += 1

y_af_norm = [auth_fail_normal.get(i, 0) for i in x]
y_af_atk  = [auth_fail_attack.get(i, 0) for i in x]

ax2.bar(x_mins, [n + a for n, a in zip(y_af_norm, y_af_atk)],
        width=bucket_size_min * 0.9, color='#0D6E6E', alpha=0.7, label='Normal auth failures')
ax2.bar(x_mins, y_af_atk,
        width=bucket_size_min * 0.9, color='#C55A11', alpha=0.9, label='Attack auth failures')
ax2.axvline(INJECTION_HOUR * 60, color='#C55A11', linestyle='--', linewidth=2,
            label='Attack start')
ax2.set_xlabel('Minutes elapsed since start of log window', fontsize=10)
ax2.set_ylabel('Auth failures per 5-min bucket', fontsize=10)
ax2.set_title('Authentication Failures Only — Attack clearly visible when isolated to this category',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Key observation:')
print('  Top panel: the attack is partially obscured by other event types (real-world challenge)')
print('  Bottom panel: when filtered to auth failures, the attack spike is obvious')
print('  Log Rate Analysis automates this filtering across ALL fields simultaneously.')


## Step 7 - Upload to Elastic Cloud


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        yield {'_index': index, '_source': doc}

print(f'Uploading {len(events):,} events to index "{INDEX_NAME}" ...')

success, errors = bulk(
    es,
    generate_actions(events, INDEX_NAME),
    chunk_size=250,
    raise_on_error=False
)

print(f'Upload complete!')
print(f'   Documents indexed : {success:,}')
if errors:
    print(f'   Errors            : {len(errors)}')
    for e in errors[:3]: print(f'      {e}')
else:
    print(f'   Errors            : 0')

es.indices.refresh(index=INDEX_NAME)
print('Index refreshed - data is now searchable in Kibana.')


## Step 8 - Verify the Upload


In [ ]:
total    = es.count(index=INDEX_NAME)['count']
injected = es.count(index=INDEX_NAME,
    body={'query': {'term': {'is_injected': True}}})['count']

agg = es.search(index=INDEX_NAME, body={
    'size': 0,
    'aggs': {
        'by_action':   {'terms': {'field': 'event.action',   'size': 10}},
        'by_category': {'terms': {'field': 'event.category', 'size': 10}},
        'by_label':    {'terms': {'field': 'injection_label','size': 5}},
    }
})

print(f'Verification summary - index "{INDEX_NAME}":')
print(f'   Total documents  : {total:,}')
print(f'   Injected events  : {injected:,}  <- the needle in the haystack')
print(f'   Normal events    : {total - injected:,}')
print()
print('Event breakdown by action:')
for b in agg['aggregations']['by_action']['buckets']:
    flag = '  <- ATTACK EVENTS HERE' if b['key'] == 'authentication_failure' else ''
    print(f'   {b["key"]:35s}  {b["doc_count"]:>5,}{flag}')
print()
print('Breakdown by injection label:')
for b in agg['aggregations']['by_label']['buckets']:
    print(f'   {b["key"]:25s}  {b["doc_count"]:>5,}')


## Step 9 - Log Rate Analysis in Kibana (Root Cause Investigation)

Now that your data is in Elastic Cloud, use Log Rate Analysis to identify
the attack signal automatically — without reading 4,000 log lines.

---

### 9a - Set Up the Data View
1. Kibana > Discover > Create data view
   - Index pattern: `aiops-lab-lograte-simulator`
   - Timestamp: `@timestamp`
   - Save the data view

---

### 9b - Run Log Rate Analysis
1. Kibana > **Machine Learning > Log Rate Analysis**
2. Select data view: `aiops-lab-lograte-simulator`
3. Set time range to cover the full 4-hour window
4. The tool will show a log rate chart. Look for the spike at the 2-hour mark.
5. Click on the spike region to select it as the 'analysis window'
6. Click **Run Analysis**
7. Wait 30-60 seconds for the analysis to complete

---

### 9c - Interpret the Results

The analysis will return a table of field-value combinations that are
**statistically overrepresented** in the spike window compared to the baseline.
Look for entries with high 'Score' values — those are the key signals.

**Expected top findings:**

| Field | Value | Why It Was Flagged |
|-------|-------|-------------------|
| `source.ip` | `91.108.4.201` | This IP appears ~0 times in baseline, 500 times in the spike |
| `user.name` | `administrator` | Unusual concentration on one account in the window |
| `event.outcome` | `failure` | Failure rate spikes dramatically in this window |
| `source.geo.country` | `Netherlands` | External country not seen in baseline traffic |
| `host.name` | `DC-PROD-01` | All attack events targeted this single host |

---

### 9d - What to Record
Answer these questions based on your Log Rate Analysis results:

> **Q1:** Which field-value pair had the highest anomaly score in the analysis?

> **Q2:** Without Log Rate Analysis, how many log lines would you need to read
> to manually identify the source IP of the attacker?

> **Q3:** This analysis took under 60 seconds. How does that compare to manual
> log investigation time for your team?

---

### 9e - Create an Alert Rule that Sends an Email

1. **Kibana > Stack Management > Rules and Connectors > Create Rule**
2. Rule type: **Elasticsearch query**
3. Configure the condition:
   - Index: `aiops-lab-lograte-simulator`
   - Query: `event.action: authentication_failure AND event.outcome: failure`
   - Threshold: count > **50** in a **5-minute** window
4. Add Action: **Elastic Cloud SMTP**
   - To: Add any email address
   - Subject: `Brute Force Alert - {{context.hits}} failures from {{context.value}}`
   - Message: Add Message bodyInclude the source IP, target account, target host, and failure count
   
5. Create the rule
6. **Test it:** Run the injection cell below to trigger a fresh burst of 100 events
7. Check Email — an Email should arrive from Elastic within 1-2 minutes

---


## Step 10 - Trigger Cell: Inject a Fresh Attack Burst to Test Your Alert Rule

Run this cell to inject a fresh burst of 100 attack events with current timestamps.
This will trigger your alert rule if it is configured correctly.

Wait 1-2 minutes after running this cell, then check **Kibana > Cases**.


In [ ]:
from datetime import datetime, timezone
import random

trigger_events = []
now_ts = datetime.now(timezone.utc)

for _ in range(100):
    ts_offset = random.uniform(0, 180)  # spread over last 3 minutes
    ts = now_ts - timedelta(seconds=ts_offset)
    trigger_events.append({
        '@timestamp':         ts.isoformat(),
        'event.action':       'authentication_failure',
        'event.category':     'authentication',
        'event.outcome':      'failure',
        'event.code':         '4625',
        'event.dataset':      'system.security',
        'host.name':          ATTACK_TARGET_HOST,
        'host.os.name':       'Windows Server 2019',
        'user.name':          ATTACK_TARGET_USER,
        'source.ip':          ATTACK_IP,
        'source.port':        random.randint(49152, 65535),
        'source.geo.country': 'Netherlands',
        'network.protocol':   'smb',
        'process.name':       '',
        'log.level':          'warning',
        'message':            (
            f'Account failed to log on. '
            f'Account: {ATTACK_TARGET_USER}, Source: {ATTACK_IP}, '
            f'Reason: Unknown user name or bad password'
        ),
        'is_injected':        True,
        'injection_label':    'alert_trigger_test'
    })

success, _ = bulk(es, generate_actions(trigger_events, INDEX_NAME),
                  chunk_size=100, raise_on_error=False)
es.indices.refresh(index=INDEX_NAME)

print(f'Injected {success} fresh attack events with current timestamps.')
print(f'Wait 1-2 minutes, then check Kibana > Cases for your auto-created incident.')
print(f'If no Case appears, verify your alert rule is active and the threshold is <= 50.')




### Lab Complete

You have:
- Generated a realistic 4-hour log stream with a hidden brute-force attack
- Visualised why the attack is hard to find in combined event volume
- Used Log Rate Analysis to identify the attack signal in under 60 seconds
- Created an automated alert rule that sends an Email without human involvement

